# MIP 입문 & EV 충전 최적화 프로젝트 초안

이 노트북은 두 가지 목적을 가집니다:
1. **MIP 기초 학습** — Gurobi / CPLEX 사용법 + 수식 조직법
2. **프로젝트 초안** — 보고서의 수식을 실제 코드로 구현

---
## 목차
- [1. MIP란 무엇인가?](#1)
- [2. Gurobi 기본 사용법 (장난감 예제)](#2)
- [3. CPLEX 기본 사용법 (동일 예제)](#3)
- [4. EV 충전 MIP 수식 정리](#4)
- [5. EV 충전 MIP 구현 (Gurobi)](#5)
- [6. 결과 시각화](#6)

---
<a id='1'></a>
## 1. MIP란 무엇인가?

### 1.1 LP → MIP 개념 흐름

| 구분 | 변수 타입 | 예시 |
|------|-----------|------|
| LP (선형 계획) | 모두 연속(실수) | 전력량 r_it |
| IP (정수 계획) | 모두 정수 | 차량 개수 |
| **MIP (혼합 정수)** | 연속 + 이진/정수 혼합 | r_it (연속) + y_it (0/1) |

### 1.2 MIP의 구성 3요소

```
min  c^T x                  ← 목적함수 (최소화)
s.t. Ax  ≤ b                ← 제약식
     x_j ≥ 0                ← 연속 변수
     x_k ∈ {0, 1}           ← 이진 변수  ← 이게 있으면 MIP!
```

### 1.3 왜 이진변수가 필요한가?

우리 문제에서 **'충전 중이냐 아니냐'** 를 표현해야 합니다.

- `y_it = 1` → 차량 i가 시간 t에 충전 중
- `y_it = 0` → 충전 안 함

이 이진변수가 있으면 **C2 제약** (충전 중일 때만 전력 공급)을 깔끔하게 표현할 수 있습니다:
```
r_it ≤ r̄_i · y_it    →   y=0이면 r=0, y=1이면 r ≤ r̄_i
```

### 1.4 MIP 풀이 방법: Branch & Bound

```
1. 이진 변수를 연속(0~1)으로 풀기 → LP Relaxation
2. 어떤 변수가 0.7처럼 분수값이면 → 두 가지 경우로 분기
   ├─ y_it = 0 으로 고정하고 다시 풀기
   └─ y_it = 1 으로 고정하고 다시 풀기
3. 정수 해가 나올 때까지 반복
```
Gurobi/CPLEX가 이 과정을 자동으로 합니다.

---
<a id='2'></a>
## 2. Gurobi 기본 사용법 — 장난감 예제

### 문제: 가장 간단한 MIP
```
max  x + y
s.t. 2x + y ≤ 10
     x + 3y ≤ 12
     x, y ≥ 0
     y ∈ {0, 1, 2, ...}  (정수)
```

In [ ]:
# Gurobi 설치 확인
# pip install gurobipy
# 주의: gurobipy는 무료로 설치되지만, 라이선스 없이는 변수 2000개 / 제약 2000개 제한
# 학교 라이선스 or 무료 Academic 라이선스: https://www.gurobi.com/academia/academic-program-and-licenses/

import gurobipy as gp
from gurobipy import GRB

print('gurobipy 버전:', gp.gurobi.version())

In [ ]:
# ── Gurobi 기본 구조 (5단계) ──────────────────────────────

# [1단계] 모델 생성
m = gp.Model('toy_example')
m.Params.OutputFlag = 1  # 0이면 솔버 로그 숨김, 1이면 출력

# [2단계] 변수 추가
#   vtype=GRB.CONTINUOUS → 연속
#   vtype=GRB.BINARY     → 이진 (0 or 1)
#   vtype=GRB.INTEGER    → 정수
x = m.addVar(vtype=GRB.CONTINUOUS, name='x', lb=0)  # x ≥ 0
y = m.addVar(vtype=GRB.INTEGER,    name='y', lb=0)  # y ≥ 0, 정수

# [3단계] 목적함수 설정
#   GRB.MAXIMIZE or GRB.MINIMIZE
m.setObjective(x + y, GRB.MAXIMIZE)

# [4단계] 제약식 추가
m.addConstr(2*x + y <= 10, name='C1')
m.addConstr(x + 3*y <= 12, name='C2')

# [5단계] 최적화 실행
m.optimize()

# 결과 출력
print('\n=== 결과 ===')
print(f'최적값 (목적함수): {m.ObjVal:.4f}')
print(f'x = {x.X:.4f}')
print(f'y = {y.X:.4f}')
print(f'상태: {m.Status}  (2=최적해 발견)')

In [ ]:
# ── Gurobi 핵심 문법 요약 ────────────────────────────────

# 변수를 여러 개 한번에 만들기 (addVars)
m2 = gp.Model('summary')

I = [0, 1, 2]          # 차량 인덱스
T = [0, 1, 2, 3, 4]    # 시간 슬롯 인덱스

# r[i, t]: 연속 변수 (2D 딕셔너리)
r = m2.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name='r')

# y[i, t]: 이진 변수 (2D 딕셔너리)
y = m2.addVars(I, T, vtype=GRB.BINARY, name='y')

# Z: 단일 연속 변수
Z = m2.addVar(vtype=GRB.CONTINUOUS, lb=0, name='Z')

# quicksum: 수식에서 Σ 표현
total_power_t0 = gp.quicksum(r[i, 0] for i in I)
print('t=0에서 총 전력 수식:', total_power_t0)

# addConstrs: 반복 제약식 한번에 추가
m2.addConstrs(
    (gp.quicksum(r[i, t] for i in I) <= Z for t in T),
    name='peak_def'
)

print('변수 수:', m2.NumVars)
print('제약 수:', m2.NumConstrs)

---
<a id='3'></a>
## 3. CPLEX 기본 사용법 — 동일 예제

CPLEX는 IBM 제품입니다. Python API는 `docplex`를 사용합니다.

```
pip install docplex
```

CPLEX 엔진 없이도 소규모(1000변수 이하)는 무료로 실행 가능합니다 (Community Edition).

In [ ]:
from docplex.mp.model import Model

# [1단계] 모델 생성
mdl = Model(name='toy_cplex')

# [2단계] 변수 추가
#   continuous_var → 연속
#   binary_var     → 이진
#   integer_var    → 정수
x = mdl.continuous_var(name='x', lb=0)
y = mdl.integer_var(name='y', lb=0)

# [3단계] 목적함수
mdl.maximize(x + y)

# [4단계] 제약식
mdl.add_constraint(2*x + y <= 10, ctname='C1')
mdl.add_constraint(x + 3*y <= 12, ctname='C2')

# [5단계] 풀기
sol = mdl.solve(log_output=True)

if sol:
    print('\n=== 결과 ===')
    print(f'최적값: {mdl.objective_value:.4f}')
    print(f'x = {x.solution_value:.4f}')
    print(f'y = {y.solution_value:.4f}')
else:
    print('해를 찾지 못했습니다.')

In [ ]:
# ── CPLEX 핵심 문법 요약 ────────────────────────────────

mdl2 = Model(name='cplex_summary')

I = [0, 1, 2]
T = [0, 1, 2, 3, 4]

# continuous_var_dict: 2D 딕셔너리
r = mdl2.continuous_var_dict([(i, t) for i in I for t in T], lb=0, name='r')

# binary_var_dict
y = mdl2.binary_var_dict([(i, t) for i in I for t in T], name='y')

Z = mdl2.continuous_var(name='Z', lb=0)

# sum: CPLEX에서 Σ 표현
total_t0 = mdl2.sum(r[i, 0] for i in I)

# add_constraints: 반복 제약
mdl2.add_constraints(
    mdl2.sum(r[i, t] for i in I) <= Z for t in T
)

print('변수 수:', mdl2.number_of_variables)
print('제약 수:', mdl2.number_of_constraints)

---
<a id='4'></a>
## 4. EV 충전 MIP 수식 정리

### 4.1 집합 & 파라미터

| 기호 | 의미 |
|------|-----------|
| I | 차량 집합 |
| T | 시간 슬롯 집합 (예: 15분 단위) |
| Δt | 슬롯 길이 (시간 단위, 예: 0.25h) |
| a_i | 차량 i 도착 슬롯 |
| d_i | 차량 i 출발 슬롯 |
| e_i | 차량 i 필요 에너지 (kWh) |
| r̄_i | 충전기 최대 충전률 (kW) |
| r̲_i | 충전기 최소 충전률 (kW) |
| C | 변압기 용량 (kW) |
| λ_t | 시간 t의 TOU 전기요금 ($/kWh) |
| μ | 수요요금 단가 ($/kW) |
| K_i | 차량 i 허용 충전 중단 횟수 |

### 4.2 결정변수

```
r_it ≥ 0          연속: 차량 i, 시간 t의 충전률 (kW)
y_it ∈ {0,1}      이진: 차량 i가 시간 t에 충전 중이면 1
Z    ≥ 0          연속: 최대 피크 전력 (kW) — 보조변수
```

### 4.3 목적함수

$$\min \sum_t \left( \lambda_t \cdot \sum_i r_{it} \cdot \Delta t \right) + \mu \cdot Z$$

- 1항: TOU 에너지 요금 (저렴한 시간대에 몰아 충전하면 절약)
- 2항: 수요요금 (피크 전력 Z를 낮추면 절약)

### 4.4 제약식

```
C1. 에너지 충족:      Σ_t r_it · Δt ≥ e_i              ∀i
C2. 온/오프 연결:     r_it ≤ r̄_i · y_it                ∀i,t
C3. 주차 시간창:      y_it = 0   (t < a_i 또는 t > d_i) ∀i,t
C4. 피크 정의:        Σ_i r_it ≤ Z                       ∀t
C5. 변압기 용량:      Σ_i r_it ≤ C                       ∀t
C6. 변수 범위:        r_it ≥ 0, y_it ∈{0,1}, Z ≥ 0

[선택 확장]
C7. 최소 충전률:      r_it ≥ r̲_i · y_it                 ∀i,t
C8. 중단 횟수 제한:   Σ_t max(y_{i,t-1} - y_it, 0) ≤ K_i ∀i
```

### 4.5 C8 선형화

C8의 `max(·,0)` 부분은 MIP에서 직접 쓸 수 없습니다. 보조변수 `s_it ≥ 0`를 도입해 선형화합니다:

```
s_it ≥ y_{i,t-1} - y_it    ∀i, t≥1
s_it ≥ 0
Σ_t s_it ≤ K_i              ∀i
```

---
<a id='5'></a>
## 5. EV 충전 MIP 구현 (Gurobi)

실제 ACN 데이터 전에, **가상의 소규모 예제**로 먼저 검증합니다.
(차량 5대, 시간 슬롯 24개 = 6시간 × 15분 단위)

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd

# ── 파라미터 설정 ─────────────────────────────────────────

DELTA_T = 0.25          # 슬롯 길이 = 15분 = 0.25시간
C_cap   = 50.0          # 변압기 용량 (kW)
mu      = 10.0          # 수요요금 단가 ($/kW)
USE_C7  = True          # 최소 충전률 제약 활성화 여부
USE_C8  = True          # 중단 횟수 제약 활성화 여부

# 시간 슬롯: 0~23 (오전 8시~오후 2시, 15분 단위)
T_slots = list(range(24))

# TOU 요금 ($/kWh) — 간단화: 피크(12~18슬롯) 높음, 나머지 낮음
# 슬롯 0 = 08:00, 슬롯 12 = 11:00
lambda_t = {t: (0.20 if 12 <= t <= 20 else 0.10) for t in T_slots}

# 차량 데이터 (가상)
# 컬럼: arrival, departure, energy_needed(kWh), max_rate(kW), min_rate(kW), max_interruptions
vehicles_data = [
    # (도착슬롯, 출발슬롯, 필요에너지, 최대충전률, 최소충전률, 최대중단횟수)
    (0,  16, 15.0, 7.2, 1.4, 2),   # 차량 0: 08:00~12:00 주차, 15kWh 필요
    (2,  20, 10.0, 6.6, 1.4, 1),   # 차량 1
    (4,  23, 20.0, 7.2, 1.4, 3),   # 차량 2
    (0,  12, 8.0,  6.6, 1.4, 2),   # 차량 3
    (8,  23, 25.0, 7.2, 1.4, 2),   # 차량 4
]

I_vehicles = list(range(len(vehicles_data)))
a = {i: vehicles_data[i][0] for i in I_vehicles}   # 도착
d = {i: vehicles_data[i][1] for i in I_vehicles}   # 출발
e = {i: vehicles_data[i][2] for i in I_vehicles}   # 필요 에너지
r_max = {i: vehicles_data[i][3] for i in I_vehicles}  # 최대 충전률
r_min = {i: vehicles_data[i][4] for i in I_vehicles}  # 최소 충전률
K = {i: vehicles_data[i][5] for i in I_vehicles}   # 최대 중단 횟수

print('차량 수:', len(I_vehicles))
print('시간 슬롯 수:', len(T_slots))
print('TOU 요금:', lambda_t)

In [ ]:
# ── MIP 모델 구축 ─────────────────────────────────────────

m = gp.Model('EV_Charging_MIP')
m.Params.OutputFlag = 1   # 솔버 로그 출력
m.Params.TimeLimit  = 60  # 최대 60초

# ── 결정변수 ──────────────────────────────────────────────

# r[i, t]: 충전률 (kW), 주차 창 안에서만 생성
r = {}
for i in I_vehicles:
    for t in T_slots:
        if a[i] <= t <= d[i]:
            r[i, t] = m.addVar(lb=0, ub=r_max[i], vtype=GRB.CONTINUOUS, name=f'r_{i}_{t}')
        else:
            r[i, t] = 0.0   # 주차 창 밖은 0으로 고정 (변수 생성 안 함)

# y[i, t]: 충전 중이면 1 (이진)
y = {}
for i in I_vehicles:
    for t in T_slots:
        if a[i] <= t <= d[i]:
            y[i, t] = m.addVar(vtype=GRB.BINARY, name=f'y_{i}_{t}')
        else:
            y[i, t] = 0.0   # 주차 창 밖은 0

# Z: 피크 전력 (kW)
Z = m.addVar(lb=0, vtype=GRB.CONTINUOUS, name='Z')

# C8용 보조변수 s[i, t] (선택)
if USE_C8:
    s = {}
    for i in I_vehicles:
        for t in T_slots[1:]:
            if a[i] <= t <= d[i]:
                s[i, t] = m.addVar(lb=0, vtype=GRB.CONTINUOUS, name=f's_{i}_{t}')

m.update()
print(f'변수 수: {m.NumVars}')
print(f'이진 변수 수: {m.NumBinVars}')

In [ ]:
# ── 목적함수 ──────────────────────────────────────────────

# TOU 에너지 요금 합 + 수요요금
tou_cost = gp.quicksum(
    lambda_t[t] * r[i, t] * DELTA_T
    for i in I_vehicles
    for t in T_slots
    if isinstance(r[i, t], gp.Var)   # 실제 변수인 경우만
)
demand_cost = mu * Z

m.setObjective(tou_cost + demand_cost, GRB.MINIMIZE)

# ── 제약식 ────────────────────────────────────────────────

# C1. 에너지 충족
for i in I_vehicles:
    m.addConstr(
        gp.quicksum(r[i, t] * DELTA_T for t in T_slots if isinstance(r[i, t], gp.Var)) >= e[i],
        name=f'C1_energy_{i}'
    )

# C2. 온/오프 연결: r_it ≤ r̄_i · y_it
for i in I_vehicles:
    for t in T_slots:
        if isinstance(r[i, t], gp.Var):
            m.addConstr(r[i, t] <= r_max[i] * y[i, t], name=f'C2_onoff_{i}_{t}')

# C4. 피크 정의: Σ_i r_it ≤ Z
for t in T_slots:
    m.addConstr(
        gp.quicksum(r[i, t] for i in I_vehicles if isinstance(r[i, t], gp.Var)) <= Z,
        name=f'C4_peak_{t}'
    )

# C5. 변압기 용량: Σ_i r_it ≤ C
for t in T_slots:
    m.addConstr(
        gp.quicksum(r[i, t] for i in I_vehicles if isinstance(r[i, t], gp.Var)) <= C_cap,
        name=f'C5_transformer_{t}'
    )

# C7. 최소 충전률: r_it ≥ r̲_i · y_it (선택)
if USE_C7:
    for i in I_vehicles:
        for t in T_slots:
            if isinstance(r[i, t], gp.Var):
                m.addConstr(r[i, t] >= r_min[i] * y[i, t], name=f'C7_minrate_{i}_{t}')

# C8. 중단 횟수 제한 (선택)
if USE_C8:
    for i in I_vehicles:
        for t in T_slots[1:]:
            if isinstance(y[i, t], gp.Var) and (i, t) in s:
                prev_y = y[i, t-1] if isinstance(y[i, t-1], gp.Var) else 0.0
                # s_it ≥ y_{i,t-1} - y_{i,t}
                m.addConstr(s[i, t] >= prev_y - y[i, t], name=f'C8a_{i}_{t}')
        # Σ_t s_it ≤ K_i
        valid_s = [s[i, t] for t in T_slots[1:] if (i, t) in s]
        if valid_s:
            m.addConstr(gp.quicksum(valid_s) <= K[i], name=f'C8b_{i}')

m.update()
print(f'제약 수: {m.NumConstrs}')

In [ ]:
# ── 최적화 실행 ───────────────────────────────────────────

m.optimize()

# 상태 확인
status_map = {2: '최적해', 3: '불가능', 5: '비유계', 9: '시간초과'}
print(f'\n풀이 상태: {status_map.get(m.Status, m.Status)}')

if m.Status in [2, 9] and m.SolCount > 0:
    print(f'총 비용 (목적함수): ${m.ObjVal:.2f}')
    print(f'피크 전력 Z*: {Z.X:.2f} kW')
    
    tou_val = sum(lambda_t[t] * (r[i,t].X if isinstance(r[i,t], gp.Var) else 0) * DELTA_T
                  for i in I_vehicles for t in T_slots)
    print(f'  TOU 에너지 요금: ${tou_val:.2f}')
    print(f'  수요요금: ${mu * Z.X:.2f}')
    print(f'MIP Gap: {m.MIPGap*100:.2f}%')

In [ ]:
# ── 스케줄 결과 추출 ──────────────────────────────────────

if m.SolCount > 0:
    schedule = []
    for i in I_vehicles:
        for t in T_slots:
            r_val = r[i, t].X if isinstance(r[i, t], gp.Var) else 0.0
            y_val = y[i, t].X if isinstance(y[i, t], gp.Var) else 0.0
            schedule.append({'vehicle': i, 'slot': t, 'rate_kW': r_val, 'charging': round(y_val)})

    df_sched = pd.DataFrame(schedule)
    
    # 차량별 충전 에너지 확인
    print('=== 차량별 충전 결과 ===')
    for i in I_vehicles:
        charged = df_sched[df_sched['vehicle']==i]['rate_kW'].sum() * DELTA_T
        print(f'  차량 {i}: {charged:.2f} kWh 충전 (필요: {e[i]} kWh)')

    # 시간별 총 전력
    load_profile = df_sched.groupby('slot')['rate_kW'].sum().reset_index()
    load_profile.columns = ['slot', 'total_kW']
    print(f'\n피크 전력: {load_profile["total_kW"].max():.2f} kW')

---
<a id='6'></a>
## 6. 결과 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

if m.SolCount > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('EV 충전 MIP 최적화 결과', fontsize=14, fontweight='bold')

    slot_labels = [f'{8 + t//4}:{(t%4)*15:02d}' for t in T_slots]

    # ── 그래프 1: 부하 프로파일 (MIP vs 비제어)
    ax1 = axes[0, 0]
    # 비제어 충전 (UC baseline): 도착 즉시 최대 충전
    uc_load = {t: 0 for t in T_slots}
    for i in I_vehicles:
        # 필요 에너지를 채울 때까지 최대 속도로 충전
        remaining = e[i]
        for t in range(a[i], d[i]+1):
            if remaining <= 0:
                break
            charge = min(r_max[i], remaining / DELTA_T)
            uc_load[t] += charge
            remaining -= charge * DELTA_T

    ax1.plot(T_slots, load_profile['total_kW'].values, 'b-o', markersize=4, label='MIP 최적 스케줄')
    ax1.plot(T_slots, [uc_load[t] for t in T_slots], 'r--s', markersize=4, label='비제어 충전 (UC)')
    ax1.axhline(y=C_cap, color='gray', linestyle=':', label=f'변압기 용량 ({C_cap}kW)')
    ax1.set_xlabel('시간')
    ax1.set_ylabel('총 전력 (kW)')
    ax1.set_title('부하 프로파일 비교')
    ax1.set_xticks(T_slots[::4])
    ax1.set_xticklabels([slot_labels[t] for t in T_slots[::4]], rotation=45)
    ax1.legend()
    ax1.grid(alpha=0.3)

    # ── 그래프 2: Gantt 차트 (차량별 충전 타임라인)
    ax2 = axes[0, 1]
    colors = plt.cm.tab10(np.linspace(0, 1, len(I_vehicles)))
    for i in I_vehicles:
        charging_slots = [t for t in T_slots if isinstance(y[i,t], gp.Var) and round(y[i,t].X) == 1]
        for t in charging_slots:
            ax2.barh(i, DELTA_T, left=t*DELTA_T, color=colors[i], alpha=0.7, edgecolor='white')
        # 주차 기간 표시
        ax2.barh(i, (d[i]-a[i])*DELTA_T, left=a[i]*DELTA_T, color=colors[i], alpha=0.15, edgecolor='gray')
    ax2.set_yticks(I_vehicles)
    ax2.set_yticklabels([f'차량 {i}' for i in I_vehicles])
    ax2.set_xlabel('시간 (h)')
    ax2.set_title('Gantt 차트 (진한색=충전 중, 연한색=주차 중)')
    ax2.grid(axis='x', alpha=0.3)

    # ── 그래프 3: TOU 요금 프로파일
    ax3 = axes[1, 0]
    ax3.bar(T_slots, [lambda_t[t] for t in T_slots], color=['tomato' if lambda_t[t] > 0.15 else 'steelblue' for t in T_slots])
    ax3.set_xlabel('시간 슬롯')
    ax3.set_ylabel('요금 ($/kWh)')
    ax3.set_title('TOU 전기요금')
    ax3.set_xticks(T_slots[::4])
    ax3.set_xticklabels([slot_labels[t] for t in T_slots[::4]], rotation=45)
    ax3.grid(alpha=0.3)

    # ── 그래프 4: 비용 비교 막대그래프
    ax4 = axes[1, 1]
    uc_tou = sum(lambda_t[t] * uc_load[t] * DELTA_T for t in T_slots)
    uc_peak = max(uc_load.values())
    uc_demand = mu * uc_peak
    uc_total = uc_tou + uc_demand

    mip_tou = tou_val
    mip_demand = mu * Z.X
    mip_total = mip_tou + mip_demand

    x_pos = [0, 1]
    ax4.bar(x_pos, [uc_tou, mip_tou], label='TOU 에너지 요금', color='steelblue')
    ax4.bar(x_pos, [uc_demand, mip_demand], bottom=[uc_tou, mip_tou], label='수요요금', color='tomato')
    ax4.set_xticks(x_pos)
    ax4.set_xticklabels(['비제어 충전\n(UC)', 'MIP 최적'])
    ax4.set_ylabel('비용 ($)')
    ax4.set_title(f'비용 비교 (절감: ${uc_total - mip_total:.2f}, {(uc_total-mip_total)/uc_total*100:.1f}%)')
    ax4.legend()
    ax4.grid(alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig('ev_mip_result.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('ev_mip_result.png 저장 완료')

---
## 7. 다음 단계: 실제 ACN 데이터 연결

위 코드의 `vehicles_data`를 실제 ACN 데이터로 교체하면 됩니다.

```python
import polars as pl

# ACN office001 데이터 로드
df = pl.scan_csv('acn_data/office001.csv', ignore_errors=True)

# 특정 날 필터 (예: 2020-06-01)
day_df = (
    df.filter(pl.col('connectionTime').str.starts_with('2020-06-01'))
      .select(['connectionTime', 'disconnectTime', 'kWhDelivered', 'stationID'])
      .collect()
)

# 시간 슬롯 변환 → a[i], d[i], e[i] 추출 → MIP에 투입
```

### 체크리스트
- [ ] Gurobi Academic 라이선스 발급
- [ ] 장난감 예제 실행 확인
- [ ] ACN 데이터 하루치 추출 → vehicles_data 변환
- [ ] 10~20대 차량 소규모 실험
- [ ] C7, C8 활성화/비활성화 비교
- [ ] 50대 이상 대규모 실험